In [1]:
!pip install -U torch torchvision torchaudio transformers langchain langchain-community langchain-huggingface langchain-qdrant qdrant-client sentence-transformers rank_bm25 pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.w

In [2]:
import torch
import torchvision
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from qdrant_client import QdrantClient
from langchain_qdrant import Qdrant as LangchainQdrant
from rank_bm25 import BM25Okapi
from typing import List, Tuple
import numpy as np
import logging

logging :
 - DEBUG: 상세한 정보, 주로 문제 진단을 위해 사용
 - INFO: 일반적인 정보
 - WARNING: 예상치 못한 일이 발생했거나 가까운 미래에 문제가 발생할 수 있는 경우
 - ERROR: 중대한 문제로 인해 소프트웨어가 일부 기능을 수행하지 못한 경우
 - CRITICAL: 프로그램 자체가 실행을 계속할 수 없는 심각한 에러
 - %(asctime)s: 시간 정보
 - %(levelname)s: 로그 레벨
 - %(message)s: 로그 메시지
 - %(filename)s: 파일 이름
 - %(funcName)s: 함수 이름

사용 이유 :
 - 디버깅 용이
 - 실행 흐름 추적
 - 정보 수집
 - 에러 처리
 - 유지보수 용이
 - 레벨별 로깅
 - 시간 정보 포함

임베딩, 검색, 재순위화, LLM 응답 생성 등 단계가 복잡함.
물론, 프로토 타입이라 불필요할 수 있으나, 09 단계에서 계속 오류가 발생하여 사용해봄.

In [3]:
# 로깅 설정
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

In [4]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son5"

In [5]:
# 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# Qdrant 클라이언트 및 벡터 스토어 초기화
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
vector_store = LangchainQdrant(
    client=client,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use QdrantVectorStore instead.
  warn_deprecated(


In [7]:
# 샘플 모델 초기화 (model gpu가 없으면 cpu 사용)
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer_config.json:   0%|          | 0.00/64.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/732k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/233k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Hugging Face의 pipeline을 사용해 텍스트 생성 파이프라인을 구성

In [8]:
# 파이프라인 생성
pipe = pipeline(
    "text-generation",  # 파이프라인 타입: 텍스트 생성
    model=model,        # 사용할 사전 훈련된 모델
    tokenizer=tokenizer,  # 사용할 토크나이저
    max_length=256,     # 생성될 텍스트의 최대 길이 (토큰 단위)
    do_sample=True,     # 확률적 샘플링 사용 (다양성 증가)
    temperature=0.7,    # 생성의 무작위성 조절 (0: 결정적, 1: 무작위)
    top_p=0.92,         # Nucleus sampling: 상위 92% 확률의 토큰만 고려
    no_repeat_ngram_size=2,  # 2-gram 반복 방지 (텍스트 반복 감소)
    device=0 if torch.cuda.is_available() else -1  # GPU 사용 가능시 GPU, 아니면 CPU 사용
)

HuggingFacePipeline을 사용해 생성된 파이프라인을 LangChain과 통합

In [9]:
# LLM 초기화
llm = HuggingFacePipeline(pipeline=pipe)

In [10]:
# 프롬프트 템플릿 정의
prompt_template = """
질문: {question}

컨텍스트: {context}

답변: """

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["question", "context"]
)

RetrievalQA 체인 :
 - 대규모 문서 컬렉션에서 관련 정보를 검색하고, 이를 바탕으로 질문에 답변하는 시스템을 구축
 - 작동 원리:
a) 질문 입력 → b) 관련 문서 검색 → c) 검색된 문서와 질문을 결합 → d) LLM을 사용해 답변 생성
 - 주요 구성 요소:
 - LLM (Language Model): 질문에 대한 답변을 생성합니다.
 - Retriever: 관련 문서를 검색하는 역할을 합니다.
 - Chain Type: 검색된 문서를 어떻게 처리할지 결정합니다.

In [11]:
# RetrievalQA 체인 생성
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # 사용할 언어 모델 (앞서 HuggingFacePipeline로 초기화한 모델)
    chain_type="stuff",  # 사용할 체인 유형 ("stuff" 메서드 사용)
    # "stuff" : 검색된 모든 문서를 하나의 컨텍스트로 결합하여 LLM에 전달(컨텍스트 길이 제한있음)
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),  # 벡터 스토어에서 상위 5개 문서 검색
    return_source_documents=True,  # 소스 문서 반환 옵션 활성화
    chain_type_kwargs={"prompt": PROMPT}  # 사용자 정의 프롬프트 템플릿 사용
)

In [12]:
# 메타데이터 추출 함수
def extract_metadata(query, categories, diseases, intents):
    query_lower = query.lower()  # 쿼리를 소문자로 변환
    # 카테고리, 질병, 의도를 쿼리에서 찾아 반환
    category = next((cat for cat in categories if cat.lower() in query_lower), None)
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)
    intent = next((intent for intent in intents if intent.lower() in query_lower), None)
    return category, disease, intent

# 고유 메타데이터 가져오기
def get_unique_metadata():
    # Qdrant에서 모든 데이터 포인트를 가져옴
    response = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=None,
        limit=10000,  # 최대 10000개의 데이터 포인트를 가져옴
        with_payload=True,  # 페이로드(메타데이터) 포함
        with_vectors=False  # 벡터는 제외
    )

    # 고유한 메타데이터 값을 저장할 집합 초기화
    categories = set()
    diseases = set()
    intents = set()

    # 각 데이터 포인트에서 메타데이터 추출 및 집합에 추가
    for point in response[0]:
        categories.add(point.payload.get("질병_카테고리", ""))
        diseases.add(point.payload.get("질병", ""))
        intents.add(point.payload.get("의도", ""))

    # 빈 문자열을 제거하고 리스트로 변환하여 반환
    return list(filter(None, categories)), list(filter(None, diseases)), list(filter(None, intents))

# 전역 변수로 고유 메타데이터 저장
DISEASE_CATEGORIES, DISEASES, INTENTS = get_unique_metadata()

In [13]:
# 쿼리 확장 함수
def expand_query(query: str) -> List[str]:
    expanded_queries = [query]  # 원본 쿼리로 리스트 초기화

    # 동의어 사전 정의
    synonyms = {
        "증상": ["징후", "병증"],
        "치료": ["처치", "요법"],
        "예방": ["방지", "예방책"]
    }

    # 동의어를 사용하여 쿼리 확장
    for word, syns in synonyms.items():
        if word in query:
            for syn in syns:
                expanded_queries.append(query.replace(word, syn))

    # 메타데이터 추출
    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)

    # 추출된 메타데이터를 사용하여 쿼리 추가 확장
    if category:
        expanded_queries.append(f"{query} {category}")
    if disease:
        expanded_queries.append(f"{query} {disease}")
    if intent:
        expanded_queries.append(f"{query} {intent}")

    # 중복 제거 후 확장된 쿼리 리스트 반환
    return list(set(expanded_queries))

Best Matching 25 :
 - 정보 검색에서 사용되는 ranking 함수
 - 텍스트 기반의 문서들을 쿼리에 대한 관련성에 따라 순위를 매김
 - TF-IDF 기반 발전된 형태
 - 문서 길이를 고려하여 긴 문서에 불이익
 - 단어 빈도에 대한 포화 효과를 적용(단어가 여러 번 나온다고 해서 무한정 점수가 올라가지 않음)

한계 :
 - 단어의 의미나 문맥을 고려하지 않음
 - 동의어나 관련 단어를 자동으로 처리하지 못함

In [14]:
# BM25를 이용한 reranking 함수
def rerank_with_bm25(query: str, documents: List[Tuple[str, float]]) -> List[Tuple[str, float]]:
    # 문서 텍스트만 추출하여 corpus 생성
    corpus = [doc[0] for doc in documents]

    # BM25 모델 초기화
    bm25 = BM25Okapi(corpus)

    # 쿼리에 대한 각 문서의 BM25 점수 계산
    scores = bm25.get_scores(query.split())

    # 문서와 점수를 짝지어 점수 기준 내림차순으로 정렬
    reranked = sorted(zip(corpus, scores), key=lambda x: x[1], reverse=True)

    # 정렬된 결과를 리스트 형태로 반환
    return [(doc, score) for doc, score in reranked]

In [15]:
# 안전한 검색 함수
def safe_similarity_search_with_score(vector_store, query, k=3):
    try:
        # 벡터 스토어에서 유사도 검색 수행
        results = vector_store.similarity_search_with_score(query, k=k)

        filtered_results = []
        for doc, score in results:
            # 문서의 내용이 유효한지 확인
            if not hasattr(doc, 'page_content') or doc.page_content is None or not doc.page_content.strip():
                # 유효하지 않은 문서는 로그를 남기고 건너뜀
                logging.warning(f"Skipping document with invalid page_content: {doc}")
                continue
            # 유효한 문서만 결과에 추가
            filtered_results.append((doc, score))

        return filtered_results

    except Exception as e:
        # 검색 중 발생한 모든 예외를 로깅하고 빈 리스트 반환
        logging.error(f"검색 중 오류 발생: {str(e)}")
        return []

In [16]:
# Qdrant 데이터 검사 함수
def inspect_qdrant_data():
    logging.info("Inspecting Qdrant data...")

    # Qdrant에서 데이터 가져오기 (최대 10개)
    response = client.scroll(
        collection_name=COLLECTION_NAME,
        limit=10,
        with_payload=True,  # 페이로드 포함
        with_vectors=False  # 벡터는 제외
    )

    # 각 데이터 포인트 검사
    for point in response[0]:
        # 데이터 포인트의 ID 로깅
        logging.info(f"ID: {point.id}")

        # 전체 페이로드 로깅
        logging.info(f"Payload: {point.payload}")

        # '답변' 또는 'page_content' 필드 확인 및 로깅
        if '답변' in point.payload:
            # '답변' 필드가 있으면 처음 100자만 로깅
            logging.info(f"답변: {point.payload['답변'][:100]}...")
        elif 'page_content' in point.payload:
            # 'page_content' 필드가 있으면 처음 100자만 로깅
            logging.info(f"Page Content: {point.payload['page_content'][:100]}...")
        else:
            # 둘 다 없으면 해당 사실을 로깅
            logging.info("Neither '답변' nor 'page_content' found in payload")

        # 구분선 추가
        logging.info("---")

In [17]:
# 검색 프로세스 디버깅 함수
def debug_search_process(query):
    # 쿼리 로깅
    logging.info(f"Debugging search process for query: {query}")

    # 메타데이터 추출
    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
    logging.info(f"메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")

    # 안전한 유사도 검색 수행
    results = safe_similarity_search_with_score(vector_store, query, k=5)
    logging.info(f"검색 결과 수: {len(results)}")

    # 각 검색 결과 로깅
    for i, (doc, score) in enumerate(results):
        logging.info(f"결과 {i+1}:")
        logging.info(f"Score: {score}")
        logging.info(f"Page Content: {doc.page_content[:100]}...")  # 처음 100자만 로깅
        logging.info(f"Metadata: {doc.metadata}")
        logging.info("---")  # 구분선

# 유사도 임계값 설정
SIMILARITY_THRESHOLD = 0.7

In [18]:
# 쿼리 처리 함수
def process_query(query):
    # 쿼리 로깅
    logging.info(f"처리 중인 쿼리: {query}")

    # 메타데이터 추출
    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
    logging.info(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")

    # 쿼리 확장
    expanded_queries = expand_query(query)
    logging.info(f"확장된 쿼리: {expanded_queries}")

    # 확장된 쿼리로 검색 수행
    all_results = []
    for eq in expanded_queries:
        results = safe_similarity_search_with_score(vector_store, eq, k=3)
        all_results.extend(results)

    # 검색 결과가 없는 경우 처리
    if not all_results:
        logging.warning("검색 결과가 없습니다.")
        return generate_llm_response(query, use_db=False)

    # 중복 제거 및 유사도 점수로 정렬
    unique_results = sorted(set(all_results), key=lambda x: x[1])

    # 최상의 유사도 계산 (Qdrant는 거리를 반환하므로 1에서 빼서 유사도로 변환)
    best_similarity = 1 - unique_results[0][1] if unique_results else 0

    # 유사도가 임계값 이상인 경우 DB 참고
    if best_similarity >= SIMILARITY_THRESHOLD:
        logging.info(f"유사도 {best_similarity:.4f}로 DB 참고")
        top_results = unique_results[:5]  # 상위 5개 결과 선택

        # BM25를 이용한 재순위화
        documents_for_reranking = [(doc.page_content, score) for doc, score in top_results]
        reranked_results = rerank_with_bm25(query, documents_for_reranking)

        # 재순위화 결과가 있고 점수가 0보다 큰 경우 처리
        if reranked_results and reranked_results[0][1] > 0:
            context = "\n".join([doc for doc, _ in reranked_results])
            return generate_llm_response(query, context=context, use_db=True, reranked_results=reranked_results)
    else:
        # 유사도가 임계값 미만인 경우 DB 미참고
        logging.info(f"유사도 {best_similarity:.4f}로 DB 미참고")

    # DB를 참고하지 않고 LLM 응답 생성
    return generate_llm_response(query, use_db=False)

In [19]:
# LLM 응답 생성 함수
def generate_llm_response(query, context=None, use_db=True, reranked_results=None):
    try:
        # DB를 참고하는 경우와 그렇지 않은 경우를 구분하여 LLM에 쿼리 전달
        if use_db and context:
            # DB에서 가져온 context를 포함하여 쿼리 수행
            result = qa_chain.invoke({"query": query, "context": context})
        else:
            # context 없이 쿼리만으로 수행
            result = qa_chain.invoke({"query": query})

        # LLM이 생성한 답변 추출
        answer = result['result']

        # 답변 출력 (DB 참고 여부에 따라 다른 메시지 출력)
        print("\nDB 참고 생성된 답변:" if use_db and context else "\nDB를 참고하지 않고 생성된 답변:")
        print(answer)

        # DB를 참고했고 재순위화된 결과가 있는 경우, 참고한 문서 정보 출력
        if use_db and reranked_results:
            print("\n참고한 문서:")
            for i, (doc, score) in enumerate(reranked_results, 1):
                print(f"{i}. BM25 점수: {score:.4f}")
                print(f"   내용: {doc[:100]}...")  # 문서 내용의 처음 100자만 출력
                print()

        return answer

    except Exception as e:
        # 오류 발생 시 로깅 및 오류 메시지 반환
        logging.error(f"LLM 응답 생성 중 오류 발생: {str(e)}")
        return f"죄송합니다. 답변을 생성하는 중에 오류가 발생했습니다: {str(e)}"

In [20]:
# 메인 함수
def langchain_based_rag_search():
    inspect_qdrant_data()  # Qdrant 데이터 구조 확인

    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ").strip()

        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        try:
            debug_search_process(query)  # 검색 프로세스 디버깅
            answer = process_query(query)
            print(answer)
        except Exception as e:
            logging.error(f"오류 발생: {str(e)}", exc_info=True)
            print(f"오류가 발생했습니다: {str(e)}")

        print("-" * 50)

if __name__ == "__main__":
    langchain_based_rag_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 통풍을 예방하는 방법을 알려줘.


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



DB를 참고하지 않고 생성된 답변:

질문: 통풍을 예방하는 방법을 알려줘.

컨텍스트: 









답변: 고퓨린 식이를 피하고, 채소와 과일 섭취를 늘리며, 규칙적인 운동, 충분한 수분 섭취, 알코올 및 고퓨린이 함유된 식품의 제한, 체중 관리가 중요합니다. 
 
 - 체중 관리: 규칙적이고 충분한 수면, 체중 조절, 규칙적 운동. 
 <출처: 건강신문>

질문: 통풍을 예방하는 방법을 알려줘.

컨텍스트: 









답변: 고퓨린 식이를 피하고, 채소와 과일 섭취를 늘리며, 규칙적인 운동, 충분한 수분 섭취, 알코올 및 고퓨린이 함유된 식품의 제한, 체중 관리가 중요합니다. 
 
 - 체중 관리: 규칙적이고 충분한 수면, 체중 조절, 규칙적 운동. 
 <출처: 건강신문>
--------------------------------------------------
질문을 입력하세요 (종료하려면 '끝내기' 입력): 다리에 염좌가 생긴 것 같아.



DB를 참고하지 않고 생성된 답변:

질문: 다리에 염좌가 생긴 것 같아.

컨텍스트: 









답변: 발목 염 좌는 발목의 인대가 과도하게 늘어나거나 찢어져 발생합니다. 발목이 삐끗하거나 넘어질 때 인대의 손상이 있는 경우가 많습니다. 주로 발목 통증, 부기, 멍, 움직임 제한 등이 동반됩니다. 증상이 심할 경우 걷기, 운동, 계단 오르내리기 등에 제한이 있습니다. 정확한 진단을 위해 MRI나 초음파 검사를 받는 것이 좋습니다.

질문: 다리에 염좌가 생긴 것 같아.

컨텍스트: 









답변: 발목 염 좌는 발목의 인대가 과도하게 늘어나거나 찢어져 발생합니다. 발목이 삐끗하거나 넘어질 때 인대의 손상이 있는 경우가 많습니다. 주로 발목 통증, 부기, 멍, 움직임 제한 등이 동반됩니다. 증상이 심할 경우 걷기, 운동, 계단 오르내리기 등에 제한이 있습니다. 정확한 진단을 위해 MRI나 초음파 검사를 받는 것이 좋습니다.
--------------------------------------------------
질문을 입력하세요 (종료하려면 '끝내기' 입력): 끝내기
검색을 종료합니다.
